# Phase VI — Head-to-head: Tarozo ordinal patterns vs multiscale level-set geometry

This notebook is the **clean, resumable Phase-VI workflow**. It compares the closest interpretable prior representation—Tarozo et al.'s 75 tie-aware two-by-two ordinal patterns—with our multiscale level-set curvature on the **same 4,000-image ArtBench pilot** and under the **same artist-disjoint nested-CV protocol**.

Primary question:

\[\boxed{\text{Does multiscale level-set curvature add information beyond tie-aware ordinal patterns?}}\]

Important: we do **not** compare our raw Macro-F1 with the published Tarozo accuracy because the corpus and validation protocol differ. Here the classifier and folds are held fixed so the changing factor is the representation.


## 0. Setup — always run first


In [ ]:
import os, sys, subprocess, shutil, zipfile, io, urllib.request, tarfile
from pathlib import Path

REPO_URL = 'https://github.com/ardominguezm/painting-geometry.git'
BRANCH = 'multiscale-corpus-analysis'
REPO_DIR = Path('/content/painting-geometry')

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

cmd = ['git','clone','--depth','1','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)]
clone = subprocess.run(cmd, text=True, capture_output=True)
if clone.returncode != 0:
    print(clone.stderr)
    raise RuntimeError('Git clone failed. Confirm that the repository is public and the branch exists.')

subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO_DIR/'requirements.txt')], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ordpy>=1.2.0'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import numpy as np, pandas as pd, ordpy

WORK = Path('/content/phase6')
INPUTS = WORK/'inputs'
RESULTS = WORK/'results'
DATA_DIR = Path('/content/artbench_data')
for p in [WORK, INPUTS, RESULTS, DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Repository cloned ✓')
print('Branch:', BRANCH)
print('Commit:', subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip())
print('ordpy:', getattr(ordpy,'__version__','unknown'))
assert hasattr(ordpy,'two_by_two_patterns')
print('SETUP OK ✓')


## 1. Upload Phase IV and IVb outputs
Upload exactly:
- `painting_geometry_phase4_artbench_pilot.zip`
- `painting_geometry_phase4b_scale_hierarchy.zip`

This cell only refreshes the **input** directory. It deliberately does **not** delete `/content/phase6/results`, so an ordinal checkpoint survives if you rerun the upload cell.


In [ ]:
from google.colab import files
uploaded = files.upload()
needed = ['painting_geometry_phase4_artbench_pilot.zip','painting_geometry_phase4b_scale_hierarchy.zip']
missing = [x for x in needed if x not in uploaded]
if missing:
    raise FileNotFoundError(f'Missing required uploads: {missing}')

if INPUTS.exists():
    shutil.rmtree(INPUTS)
P4 = INPUTS/'phase4'
P4B = INPUTS/'phase4b'
P4.mkdir(parents=True)
P4B.mkdir(parents=True)

for name, dest in [(needed[0],P4),(needed[1],P4B)]:
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as zf:
        zf.extractall(dest)
    print('Extracted ✓', name)

def one_file(root, name):
    hits = list(root.rglob(name))
    if len(hits) != 1:
        raise RuntimeError(f'Expected exactly one {name} below {root}; found {hits}')
    return hits[0]

FEATURES = one_file(P4, 'artbench_pilot_features.csv')
OOF_ALL = one_file(P4B, 'artbench10_all_phase4b_oof_predictions.csv')
OOF_W8 = one_file(P4B, 'artbench10_wikiart8_phase4b_oof_predictions.csv')
feat = pd.read_csv(FEATURES)
print('Pilot matrix:', feat.shape)
print('Styles:', feat['style'].nunique())
print('Artists:', feat['artist'].nunique())
assert len(feat) == 4000
print('PHASE IV / IVb INPUTS READY ✓')


## 2. Obtain the ArtBench-10 256×256 ImageFolder
Ordinal patterns are recomputed from pixels. The cell first tries Kaggle's single-file download and falls back to the official ArtBench archive. If the archive is already present in this runtime it is reused.


In [ ]:
import kagglehub
KAGGLE_HANDLE = 'alexanderliao/artbench10'
TAR_NAME = 'artbench-10-imagefolder-split.tar'
EXTRACT_DIR = DATA_DIR/'imagefolder'

existing_train = list(EXTRACT_DIR.rglob('train')) if EXTRACT_DIR.exists() else []
if existing_train:
    print('ArtBench already extracted ✓', EXTRACT_DIR)
else:
    tar_path = None
    for candidate in [TAR_NAME, f'256X256/{TAR_NAME}', f'data/256X256/{TAR_NAME}', f'ArtBench-10/data/256X256/{TAR_NAME}']:
        try:
            print('Trying Kaggle file:', candidate)
            p = Path(kagglehub.dataset_download(KAGGLE_HANDLE, path=candidate))
            if p.exists() and p.is_file():
                tar_path = p
                print('Downloaded ✓', p)
                break
        except Exception as exc:
            print('  not found:', type(exc).__name__)

    if tar_path is None:
        tar_path = DATA_DIR/TAR_NAME
        official = 'https://artbench.eecs.berkeley.edu/files/artbench-10-imagefolder-split.tar'
        print('Downloading official ArtBench archive (~1.85 GB)...')
        subprocess.run(['wget','-c',official,'-O',str(tar_path)], check=True)

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    print('Extracting ArtBench...')
    with tarfile.open(tar_path) as tf:
        try:
            tf.extractall(EXTRACT_DIR, filter='data')
        except TypeError:
            tf.extractall(EXTRACT_DIR)

train_hits = list(EXTRACT_DIR.rglob('train'))
test_hits = list(EXTRACT_DIR.rglob('test'))
if not train_hits or not test_hits:
    raise RuntimeError('Could not locate extracted ArtBench train/test folders.')
print('IMAGE ARCHIVE READY ✓')


## 3. Extract Tarozo-style ordinal representations — accelerated and resumable
For the same 4,000 paintings we compute `OP75`, `OP11`, `OP24`, and the complexity–entropy pair `(H,C)`.

The extractor uses a vectorized NumPy implementation for the 2×2 ranking step, but **before touching the corpus it must reproduce ordpy exactly on tie-rich synthetic images**. A checkpoint is written every 100 paintings and automatically reused.


In [ ]:
ENRICHED = RESULTS/'artbench_pilot_features_with_ordinal.csv'
CHECKPOINT = ENRICHED.with_suffix('.ordinal_checkpoint.csv')

if CHECKPOINT.exists():
    try:
        print('Existing checkpoint rows:', len(pd.read_csv(CHECKPOINT)))
    except Exception as exc:
        print('Checkpoint exists but could not be read:', exc)

cmd = [
    sys.executable, '-u', str(REPO_DIR/'scripts'/'extract_tarozo_ordinal_features.py'),
    '--features', str(FEATURES),
    '--dataset-root', str(EXTRACT_DIR),
    '--output', str(ENRICHED),
    '--checkpoint-every', '100',
]
print('Running accelerated ordinal extractor...')
proc = subprocess.run(cmd)
if proc.returncode != 0:
    raise RuntimeError(f'Ordinal extractor failed with exit code {proc.returncode}. The script traceback is shown immediately above this message.')
print('ORDINAL EXTRACTION COMPLETE ✓')


## 4. Scientific sanity checks


In [ ]:
df = pd.read_csv(ENRICHED)
print('Enriched matrix:', df.shape)
counts = {
    'OP75': sum(c.startswith('ord75__') for c in df.columns),
    'OP11': sum(c.startswith('ord11__') for c in df.columns),
    'OP24': sum(c.startswith('ord24__') for c in df.columns),
    'HC': sum(c.startswith('ordhc__') for c in df.columns),
}
print(counts)
assert len(df) == 4000, f'Expected 4000 paintings, got {len(df)}'
assert counts == {'OP75':75,'OP11':11,'OP24':24,'HC':2}, counts
for c in ['ordmeta__sum75','ordmeta__sum11','ordmeta__sum24']:
    err = float(np.max(np.abs(df[c].to_numpy()-1.0)))
    print(c, 'max |sum-1| =', err)
    assert err < 1e-8
print('Mean tie-pattern mass:', float(df['ordmeta__tie_pattern_mass'].mean()))
print('Mean [0000] probability:', float(df['ordmeta__type_A_0000'].mean()))
failure_file = ENRICHED.with_suffix('.failures.csv')
if failure_file.exists():
    fail = pd.read_csv(failure_file)
    if len(fail):
        display(fail.head(20))
        raise RuntimeError(f'{len(fail)} images failed ordinal extraction.')
print('ALL ORDINAL SANITY CHECKS PASSED ✓')


## 5. Artist-disjoint head-to-head
Prespecified primary contrasts:

\[\Delta_1=F_1(OP75+K40)-F_1(OP75)\]

and the dimension-matched version

\[\Delta_2=F_1[(OP75+K40)_{k=40}]-F_1[(OP75)_{k=40}].\]

For WikiArt-8 we additionally test whether curvature adds information after both the strong 90-feature appearance baseline and OP75 are already present.


In [ ]:
EXP = RESULTS/'head_to_head'
EXP.mkdir(parents=True, exist_ok=True)
cmd = [
    sys.executable, '-u', str(REPO_DIR/'scripts'/'run_ordinal_geometry_head_to_head.py'),
    '--features', str(ENRICHED),
    '--output-dir', str(EXP),
    '--phase4b-all-oof', str(OOF_ALL),
    '--phase4b-wiki8-oof', str(OOF_W8),
    '--outer-folds', '5', '--inner-folds', '3', '--n-jobs', '-1',
    '--metric-boot', '2000', '--delta-boot', '5000',
]
proc = subprocess.run(cmd)
if proc.returncode != 0:
    raise RuntimeError(f'Head-to-head model run failed with exit code {proc.returncode}. See script traceback above.')
print('HEAD-TO-HEAD COMPLETE ✓')


## 6. Inspect the primary results


In [ ]:
res = pd.read_csv(EXP/'phase6_head_to_head_results.csv')
deltas = pd.read_csv(EXP/'phase6_head_to_head_deltas.csv')
display(res.sort_values(['dataset','macro_f1_oof'], ascending=[True,False]))
print('PRIMARY GEOMETRY-INCREMENT CONTRASTS')
primary = deltas[deltas['primary_head_to_head'].astype(bool)].copy()
display(primary)
print('All paired contrasts')
display(deltas)


## 7. Package outputs
Download the ZIP and upload it back to the chat. Interpretation will be based on artist-group bootstrap intervals and the prespecified paired contrasts, not raw accuracy alone.


In [ ]:
OUT_ZIP = Path('/content/painting_geometry_phase6_ordinal_head_to_head.zip')
if OUT_ZIP.exists():
    OUT_ZIP.unlink()
with zipfile.ZipFile(OUT_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS.rglob('*'):
        if p.is_file():
            zf.write(p, p.relative_to(RESULTS))
print('Output ZIP:', OUT_ZIP)
print('Size MB:', OUT_ZIP.stat().st_size/1e6)
files.download(str(OUT_ZIP))
